# 分支与 Warp 发散

分支是 SIMT 的能力，也是它最大的性能陷阱。本节讲清发散的机理与对策，以及"多线程写同一位置"的正解——原子操作。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 分支发散的机理

同一 Warp 的线程按共同指令流执行。若编译结果保留了发散分支，不同路径通常按活动线程掩码分别执行；下面用忽略调度开销的简化模型说明：

```text
if (x % 2 == 0): y = a分支计算     # Warp 内 18 个线程走这里
else:            y = b分支计算     # 其余 14 个走这里
# 硬件先执行 a 分支（14 个线程空等），再执行 b 分支（18 个空等）
# 在这个简化模型中，执行时间按两条路径之和估算；实际时间还受编译转换和调度影响
```

对于无跨线程依赖的分支，发散保留各线程的分支语义，但会降低效率；分支中的屏障、共享数据依赖仍须单独满足同步规则。发散段越短、占比越小，代价越低。


## 2. 对策

1. **区分残缺 Warp 与分支发散**：线程数取 32 的整数倍可避免最后一个 Warp 不满，但不能消除 Warp 内线程走不同分支的问题；
2. **把分支改写为无分支表达**：在有限数值且两侧表达式均合法时，可使用等价的 min/max 等表达式，例如正数翻倍可写成 `x + pl.simt.max(x, 0.0)`；不要假定比较结果支持直接 cast 为 FP32；NaN/Inf、越界加载等不能靠乘零屏蔽——SIMD 世界"用掩码代替分支"的思路在 SIMT 同样适用；
3. **让同 Warp 线程尽量走同路**：按数据规律排布线程（如按 block 对齐数据段），减少 Warp 内部分歧；
4. **接受必要的发散**：算法本质分支（如稀疏选择）就是 SIMT 的适用场景，此时收益大于发散代价——第 9 章会看到"何时该容忍发散"的定量判断。


## 3. 原子更新：多线程写同一位置

`读取后写回`（`a[i,j] = a[i,j] + delta`）**不具备原子性**：两个线程同时读旧值、各自加、先后写回，会丢失一次更新。多线程操作同一元素用 **simt 原子操作族**：

```python
pl.simt.atomic_add(a, [i, j], delta)    # 原子加（还有 atomic_max 等）
```

三条边界：

1. 原子操作**不提供跨 Block 屏障**，也不保证其他地址的数据已就绪；
2. 原子操作解决"同位置竞争"，不解决"跨线程顺序"——顺序依赖靠 syncthreads / 事件；
3. 完整类型与返回值规则见原子操作 API 文档。

典型场景：直方图（多线程累加同一 bin）、块内共享归约（softmax_simt 的 atomic_max/atomic_add）及分阶段 GM 归约、计数器。


### 四组实验只改变两件事

输入包含相同数量的 +1 和 −1。一组把正负分在连续两半，另一组交替排列；两版程序的输出分别都应为正位置 2、负位置 −1。先做全量精度校验，再预热、重复计时，不能只比较耗时而忽略表达式变了。

本章脚本报告 Host 墙钟每次调用耗时，包含下发与设备工作；小规模情况下，下发开销可能盖过分歧差异。若需解释 Warp 的实际执行，需结合第 8 章设备侧 profiling。编译器可能转换简单分支，因此“交替输入必慢两倍”不是可预填的实验结论。无显式分支版输入限定为有限值，不能推广到 Inf/NaN。


## 4. 课后练习

1. （填空）Warp 内 20 个线程走 A 分支、12 个走 B 分支，A/B 各需 1 拍时，Warp 完成这段需要 ______ 拍（锁步执行、逐分支串行）。
2. （选择）多个线程对同一 Tile 元素 `+=1`，正确做法：
   A. `a[i,j] = a[i,j] + 1`<br/>
   B. `pl.simt.atomic_add(a, [i,j], 1)`<br/>
   C. 加 syncthreads 后再写
3. （实践）把 `if x > 0: y = x else: y = 0`（ReLU）改写为无分支表达（限定有限 FP32 输入，提示：pl.simt.max(x, 0.0)，不要直接将布尔比较结果 cast 为 FP32）。

In [ ]:
!cat ./answer/06.05_answer.txt

## 5. 本节小结

- 分支发散 = Warp 逐分支串行，独立分支语义保持，屏障与共享依赖仍须检查；
- 对策：**无分支改写、同路排布、必要时接受**；32 对齐只解决残缺 Warp；
- 同位置竞争用**原子操作**；原子不管跨 Block 屏障与顺序依赖。